<h1 style="text-align: center; color: #1a5276;">Evaluación de solicitud de crédito bancario</h1>
<p style="text-align: justify; font-size: 13px; color: #333333;">
Sistema inteligente para la evaluación automatizada de solicitudes de crédito bancario a partir del cálculo de apalancamiento financiero e historial crediticio.
</p>

<h2><span style="color: #2980b9;">1. Diseño del Agente: Ficha PEAS</span></h2>

<table style="width: 100%; border-collapse: collapse; margin-top: 8px; font-size: 13px;">
  <thead>
    <tr style="background-color: #f2f4f4; text-align: left;">
      <th style="border: 1px solid #d5dbdb; padding: 7px; width: 22%;">Elemento PEAS</th>
      <th style="border: 1px solid #d5dbdb; padding: 7px;">Especificación Técnica</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid #d5dbdb; padding: 7px;"><b>Percepción (S)</b></td>
      <td style="border: 1px solid #d5dbdb; padding: 7px;"><code>ingreso_mensual</code> (S/), <code>monto_solicitado</code> (S/) y <code>tiene_historial_moroso</code> (Booleano).</td>
    </tr>
    <tr>
      <td style="border: 1px solid #d5dbdb; padding: 7px;"><b>Acciones (A)</b></td>
      <td style="border: 1px solid #d5dbdb; padding: 7px;"><code>aprobar</code>, <code>aprobar con condiciones</code>, <code>rechazar</code>.</td>
    </tr>
    <tr>
      <td style="border: 1px solid #d5dbdb; padding: 7px;"><b>Entorno (E)</b></td>
      <td style="border: 1px solid #d5dbdb; padding: 7px;">Plataforma de evaluación crediticia y base de datos de riesgo.</td>
    </tr>
    <tr>
      <td style="border: 1px solid #d5dbdb; padding: 7px;"><b>Objetivo</b></td>
      <td style="border: 1px solid #d5dbdb; padding: 7px;">Maximizar colocación de créditos minimizando el riesgo de impago.</td>
    </tr>
  </tbody>
</table>

<h2><span style="color: #2980b9;">2. Justificación de las reglas de decisión</span></h2>
<ol style="font-size: 13px; color: #333333; line-height: 1.6; margin-left: 20px;">
  <li><b>Historial moroso activo:</b> Rechazo inmediato por alta probabilidad estadística de impago.</li>
  <li><b>Relación monto/ingreso &le; 3.0:</b> Aprobación directa. El cliente tiene flujo de caja suficiente para la amortización.</li>
  <li><b>Relación 3.0 &lt; monto/ingreso &le; 6.0:</b> Aprobación con condiciones (requiere aval o garantía). Es un endeudamiento moderado.</li>
  <li><b>Relación monto/ingreso &gt; 6.0:</b> Rechazo por sobreendeudamiento crítico. Supera la capacidad máxima de pago.</li>
</ol>

In [1]:
def agente_credito(ingreso_mensual, monto_solicitado, tiene_historial_moroso):
    if ingreso_mensual <= 0:
        return "rechazar", "ingreso mensual invalido o menor a cero"
    
    relacion = monto_solicitado / ingreso_mensual
    
    if tiene_historial_moroso:
        return "rechazar", f"relacion {relacion:.2f} con historial moroso activo"
    
    if relacion <= 3.0:
        return "aprobar", f"capacidad optima (relacion {relacion:.2f} <= 3.0)"
    elif relacion <= 6.0:
        return "aprobar con condiciones", f"endeudamiento moderado (relacion {relacion:.2f})"
    else:
        return "rechazar", f"sobreendeudamiento critico (relacion {relacion:.2f} > 6.0)"

# Matriz de Pruebas (Casos de frontera)
casos_prueba = [
    (4000, 12000, False),  # Frontera exacta aprobar (relacion 3.0)
    (3000, 12000, False),  # Intermedio condicionar (relacion 4.0)
    (2000, 12000, False),  # Frontera exacta condicionar (relacion 6.0)
    (2500, 20000, False),  # Sobreendeudamiento (relacion 8.0)
    (6000, 6000, True),    # Morosidad activa (relacion 1.0)
]

print("--- RESULTADOS DE MATRIZ DE PRUEBAS ---")
for i, (ing, mon, mor) in enumerate(casos_prueba, 1):
    decision, motivo = agente_credito(ing, mon, mor)
    print(f"Caso {i} | S/ {ing:<5} | S/ {mon:<6} | Moroso: {str(mor):<5} -> {decision.upper()} ({motivo})")

--- RESULTADOS DE MATRIZ DE PRUEBAS ---
Caso 1 | S/ 4000  | S/ 12000  | Moroso: False -> APROBAR (capacidad optima (relacion 3.00 <= 3.0))
Caso 2 | S/ 3000  | S/ 12000  | Moroso: False -> APROBAR CON CONDICIONES (endeudamiento moderado (relacion 4.00))
Caso 3 | S/ 2000  | S/ 12000  | Moroso: False -> APROBAR CON CONDICIONES (endeudamiento moderado (relacion 6.00))
Caso 4 | S/ 2500  | S/ 20000  | Moroso: False -> RECHAZAR (sobreendeudamiento critico (relacion 8.00 > 6.0))
Caso 5 | S/ 6000  | S/ 6000   | Moroso: True  -> RECHAZAR (relacion 1.00 con historial moroso activo)
